[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/40.%20The%20Port%20Capacity%20%26%20Expansion%20Timing%20Problem/P40-Tier-8_executed.ipynb)

---

# 40. The Port Capacity and Expansion Timing Problem
## Tier 8: The Value-Aligned and Ethical Framework

### Goal

Evaluate a finite set of port expansion alternatives by combining stakeholder-weighted economic, environmental, and social utility with ethical assessment scores. The model selects the alternative with the highest combined score and compares it with the alternative having the highest net present value (NPV).

### Inputs and outputs

The inputs are the stakeholder preference weights, ethical scoring parameters, and the attributes of four illustrative expansion alternatives. The outputs are each alternative's ethical scores and aggregate utility, the highest-scoring alternative under the value-aligned rule, and a comparison with the highest-NPV alternative.

### Scope and modeling assumptions

This is a deterministic, finite-alternative decision model, not a construction schedule or a full-scale social-welfare analysis. NPV and investment are expressed in millions of dollars (USD millions); carbon footprint is in tonnes of CO₂ per year; and job counts are numbers of jobs. The utility calculation uses explicitly stated monetary proxy values to put unlike effects on a common illustrative scale. Those proxies and stakeholder weights are assumptions for this example, not empirically estimated values.

Ethical considerations enter the selection rule as a score multiplier, not as hard feasibility constraints. Some fields in `EthicalConstraints` document policy topics that this dataset cannot operationalize: it contains no local-benefit denominator, wage data, or emissions pathway through time. Accordingly, the model does not claim to enforce a community-benefit share, a minimum wage, or a carbon-neutrality deadline. Environmental justice and labor equity are represented by the particular scoring proxies defined in the code.

### Method

For each alternative, the notebook calculates four scores in the range $[0,1]$, calculates utility for each stakeholder, and sums the stakeholder utilities. It then multiplies total utility by a factor based on the mean ethical score. The choice is the alternative with the greatest combined score. This is a transparent weighted-scoring method over the supplied alternatives; it does not search over unlisted projects or prove that a real-world project is ethically acceptable.

### Why this tier matters

A project with the highest NPV need not perform best against environmental, workforce, or community considerations. Including these considerations makes trade-offs visible, while the model's assumptions and limitations make clear which judgments still require stakeholder participation, evidence, and governance.

## 1. Set up the model

The implementation uses data classes to keep each stakeholder's preferences, the policy parameters, and each project's attributes together. This structure makes the inputs inspectable and gives later calculations a consistent set of named fields. The random seeds are retained for reproducibility if stochastic analysis is added to the surrounding workflow; the calculations in this notebook are deterministic.

In [ ]:
from dataclasses import dataclass
import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Preserve reproducibility for any random operations in the workflow.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Value-aligned and ethical framework initialized.")

## 2. Represent stakeholders, policy parameters, and alternatives

A stakeholder group is represented by weights for economic, environmental, and social considerations, along with a risk-tolerance parameter and a time-preference rate. An expansion alternative stores the data used by the scoring and utility calculations. Keeping these concepts separate allows the same project to be assessed from several stakeholder perspectives.

In [ ]:
@dataclass
class StakeholderGroup:
    """Preferences used to calculate one stakeholder group's utility."""

    name: str
    economic_weight: float
    environmental_weight: float
    social_weight: float
    risk_tolerance: float
    time_preference: float


@dataclass
class EthicalConstraints:
    """Policy parameters, some of which require data absent from this example."""

    environmental_justice_ratio: float = 0.15
    labor_displacement_ratio: float = 0.8
    community_benefit_ratio: float = 0.25
    intergenerational_debt_ratio: float = 0.4
    minimum_wage_standard: float = 1.0
    carbon_neutrality_timeline: int = 2050


@dataclass
class ExpansionAlternative:
    """An expansion option and its economic, environmental, and social attributes."""

    name: str
    investment_cost: float
    capacity_increase: float
    technology_level: str
    implementation_years: int
    npv: float = 0.0
    annual_revenue: float = 0.0
    job_creation: int = 0
    job_displacement: int = 0
    carbon_footprint: float = 0.0
    air_quality_impact: float = 0.0
    water_usage: float = 0.0
    land_use: float = 0.0
    community_benefits: float = 0.0
    local_employment: int = 0
    training_opportunities: int = 0
    accessibility_improvements: float = 0.0
    environmental_justice_score: float = 0.0
    labor_equity_score: float = 0.0
    intergenerational_score: float = 0.0
    transparency_score: float = 0.0


class ValueAlignedOptimizer:
    """Score and compare a finite set of port expansion alternatives."""

    def __init__(self):
        self.stakeholders = self._initialize_stakeholders()
        self.constraints = EthicalConstraints()
        self.alternatives: List[ExpansionAlternative] = []

    def _initialize_stakeholders(self) -> List[StakeholderGroup]:
        """Return the illustrative stakeholder groups and preference weights."""
        return [
            StakeholderGroup("Port Authority", 0.30, 0.15, 0.15, 0.6, 0.08),
            StakeholderGroup("Terminal Operators", 0.25, 0.10, 0.10, 0.7, 0.08),
            StakeholderGroup("Labor Unions", 0.15, 0.25, 0.25, 0.8, 0.05),
            StakeholderGroup("Local Community", 0.10, 0.30, 0.30, 0.9, 0.04),
            StakeholderGroup("Environmental Groups", 0.10, 0.35, 0.20, 0.9, 0.02),
            StakeholderGroup("Shipping Customers", 0.15, 0.05, 0.05, 0.5, 0.10),
        ]


print("Stakeholder, policy-parameter, and alternative data structures are defined.")

## 3. Generate and assess the alternatives

The alternatives are the choices available to this finite decision model. Their attributes are illustrative inputs, not predictions. After creating them, the method calculates the four ethical assessment scores so that every alternative is evaluated using the same rules.

In [ ]:
def generate_alternatives(self) -> List[ExpansionAlternative]:
    """Create the example alternatives and calculate their ethical scores."""
    alternatives = [
        ExpansionAlternative(
            name="Minimal Expansion",
            investment_cost=200,
            capacity_increase=0.15,
            technology_level="manual",
            implementation_years=2,
            npv=450,
            annual_revenue=50,
            job_creation=120,
            job_displacement=15,
            carbon_footprint=5000,
            air_quality_impact=20,
            water_usage=0.5,
            land_use=10,
            community_benefits=15,
            local_employment=80,
            training_opportunities=20,
            accessibility_improvements=0.3,
        ),
        ExpansionAlternative(
            name="Moderate Automation",
            investment_cost=400,
            capacity_increase=0.40,
            technology_level="semi_automated",
            implementation_years=3,
            npv=580,
            annual_revenue=80,
            job_creation=200,
            job_displacement=95,
            carbon_footprint=8000,
            air_quality_impact=35,
            water_usage=0.8,
            land_use=15,
            community_benefits=25,
            local_employment=140,
            training_opportunities=60,
            accessibility_improvements=0.6,
        ),
        ExpansionAlternative(
            name="Comprehensive Green",
            investment_cost=650,
            capacity_increase=0.60,
            technology_level="fully_automated",
            implementation_years=4,
            npv=578,
            annual_revenue=95,
            job_creation=280,
            job_displacement=95,
            carbon_footprint=6000,
            air_quality_impact=25,
            water_usage=0.6,
            land_use=20,
            community_benefits=45,
            local_employment=200,
            training_opportunities=100,
            accessibility_improvements=0.8,
        ),
        ExpansionAlternative(
            name="Maximum Capacity",
            investment_cost=800,
            capacity_increase=0.85,
            technology_level="fully_automated",
            implementation_years=5,
            npv=634,
            annual_revenue=120,
            job_creation=350,
            job_displacement=230,
            carbon_footprint=10000,
            air_quality_impact=50,
            water_usage=1.0,
            land_use=25,
            community_benefits=35,
            local_employment=180,
            training_opportunities=80,
            accessibility_improvements=0.7,
        ),
    ]

    for alternative in alternatives:
        alternative.environmental_justice_score = (
            self._calculate_environmental_justice(alternative)
        )
        alternative.labor_equity_score = self._calculate_labor_equity(alternative)
        alternative.intergenerational_score = (
            self._calculate_intergenerational_equity(alternative)
        )
        alternative.transparency_score = self._calculate_transparency(alternative)

    self.alternatives = alternatives
    return alternatives


ValueAlignedOptimizer.generate_alternatives = generate_alternatives

print("Alternative generation is ready.")

## 4. Define the ethical assessment scores

A score is a compact indicator: $1$ represents full performance under the specified proxy, and $0$ represents the bottom of that proxy's scale. These scores are not substitutes for measured impact assessments. They make the assumptions used in this example explicit and allow the alternatives to be compared consistently.

For environmental justice, let $I$ be a proxy for total impact and let $I_v=0.2I$ be the assumed portion affecting vulnerable communities. The score is

$$E_{\mathrm{env}} = \begin{cases} 1, & I_v \leq r_{\mathrm{env}}I, \\ \max\left(0, 1 - \left(\frac{I_v}{r_{\mathrm{env}}I} - 1\right)\right), & I_v > r_{\mathrm{env}}I, \end{cases}$$

where $r_{\mathrm{env}}$ is `environmental_justice_ratio`. The assumed vulnerable-community share is fixed at $0.2$ in this example. The labor-equity proxy compares displacement with the allowed amount, $r_{\mathrm{labor}}J_c + 0.5T$, where $J_c$ is jobs created and $T$ is training opportunities. The intergenerational proxy compares current costs with a share of future benefits; it assumes a carbon value of $50$ USD per tonne and a 20-year community-benefit period. Transparency is a heuristic based on technology, implementation time, and investment scale.

These are soft scores. In particular, a score below $1$ does not make an alternative infeasible; its effect enters the selection rule in the next section.

In [ ]:
def _calculate_environmental_justice(
    self, alternative: ExpansionAlternative
) -> float:
    """Score a proxy for the share of impacts affecting vulnerable communities."""
    total_impact = (
        alternative.air_quality_impact
        + alternative.carbon_footprint / 1000
        + alternative.water_usage * 10
    )
    if total_impact <= 0:
        return 1.0

    vulnerable_impact = total_impact * 0.2
    permitted_impact = self.constraints.environmental_justice_ratio * total_impact
    if vulnerable_impact <= permitted_impact:
        return 1.0

    return max(0.0, 1.0 - (vulnerable_impact / permitted_impact - 1.0))


def _calculate_labor_equity(
    self, alternative: ExpansionAlternative
) -> float:
    """Score displacement relative to job creation and training capacity."""
    retraining_capacity = alternative.training_opportunities * 0.5
    max_allowed_displacement = (
        self.constraints.labor_displacement_ratio * alternative.job_creation
        + retraining_capacity
    )
    if max_allowed_displacement <= 0:
        return 1.0 if alternative.job_displacement <= 0 else 0.0
    if alternative.job_displacement <= max_allowed_displacement:
        return 1.0

    return max(
        0.0,
        1.0 - (alternative.job_displacement / max_allowed_displacement - 1.0),
    )


def _calculate_intergenerational_equity(
    self, alternative: ExpansionAlternative
) -> float:
    """Compare illustrative current costs with a share of future benefits."""
    carbon_cost_millions = alternative.carbon_footprint * 50 / 1_000_000
    current_costs = alternative.investment_cost + carbon_cost_millions
    future_benefits = alternative.npv + alternative.community_benefits * 20
    permitted_costs = (
        self.constraints.intergenerational_debt_ratio * future_benefits
    )

    if permitted_costs <= 0:
        return 1.0 if current_costs <= 0 else 0.0
    if current_costs <= permitted_costs:
        return 1.0

    return max(0.0, 1.0 - (current_costs / permitted_costs - 1.0))


def _calculate_transparency(
    self, alternative: ExpansionAlternative
) -> float:
    """Apply a heuristic transparency score based on project complexity."""
    technology_complexity = {
        "manual": 0.2,
        "semi_automated": 0.5,
        "fully_automated": 0.8,
    }
    complexity = technology_complexity.get(alternative.technology_level, 0.5)
    time_complexity = min(alternative.implementation_years / 5.0, 1.0)
    investment_complexity = min(alternative.investment_cost / 800.0, 1.0)
    average_complexity = (
        complexity + time_complexity + investment_complexity
    ) / 3
    return max(0.3, 1.0 - average_complexity * 0.5)


ValueAlignedOptimizer._calculate_environmental_justice = (
    _calculate_environmental_justice
)
ValueAlignedOptimizer._calculate_labor_equity = _calculate_labor_equity
ValueAlignedOptimizer._calculate_intergenerational_equity = (
    _calculate_intergenerational_equity
)
ValueAlignedOptimizer._calculate_transparency = _calculate_transparency

print("Ethical assessment score calculations are ready.")

## 5. Calculate stakeholder utility and select an alternative

Utility combines the modeled economic, environmental, and social effects for one stakeholder. In this illustrative model, all components are expressed as monetary-equivalent values in USD millions using the proxy rates stated in the code. A stakeholder's time preference applies a one-period discount factor, $1/(1+t)$, where $t$ is that stakeholder's time-preference rate. Risk tolerance scales a small investment-related adjustment; it is a model input, not a measured probability of project failure.

For stakeholder $s$ and alternative $a$, the model calculates

$$U_{s,a} = \left(w^E_s N_a - w^{\mathrm{env}}_s C_a + w^S_s(B_a-D_a) - \frac{I_a}{1000}r_s\right)\frac{1}{1+t_s},$$

where $N_a$ is NPV, $C_a$ is the environmental-cost proxy, $B_a$ and $D_a$ are social-benefit and displacement-cost proxies, $I_a$ is investment, and $w^E_s$, $w^{\mathrm{env}}_s$, and $w^S_s$ are the stakeholder's weights. The aggregate utility is $U_a=\sum_s U_{s,a}$. The mean of the four ethical scores is $E_a$, and the selection score is

$$S_a = U_a(0.7+0.3E_a).$$

The coefficients preserve a weighted-scoring approach: aggregate utility remains the main component, while ethical performance adjusts the score. The selected option maximizes $S_a$ among the listed alternatives. If scores tie within a numerical tolerance, all tied alternatives are reported; the first in input order is returned as the representative choice.

In [ ]:
def calculate_stakeholder_utility(
    self,
    alternative: ExpansionAlternative,
    stakeholder: StakeholderGroup,
) -> float:
    """Calculate one stakeholder's illustrative monetary-equivalent utility."""
    economic_utility = alternative.npv * stakeholder.economic_weight

    # Convert the environmental proxy costs from USD to USD millions.
    environmental_cost = (
        alternative.carbon_footprint * 0.05
        + alternative.air_quality_impact * 1000
        + alternative.water_usage * 100000
    ) / 1_000_000
    environmental_utility = (
        -environmental_cost * stakeholder.environmental_weight
    )

    # Community benefits are in USD millions per year; the remaining terms
    # use the illustrative USD-million-per-unit proxy rates shown here.
    social_benefits = (
        alternative.community_benefits
        + alternative.local_employment * 0.05
        + alternative.accessibility_improvements * 10
    )
    social_cost = alternative.job_displacement * 0.08
    social_utility = (
        social_benefits - social_cost
    ) * stakeholder.social_weight

    risk_adjustment = (
        -(alternative.investment_cost / 1000) * stakeholder.risk_tolerance
    )
    time_factor = 1.0 / (1.0 + stakeholder.time_preference)

    return (
        economic_utility
        + environmental_utility
        + social_utility
        + risk_adjustment
    ) * time_factor


def _analyze_stakeholder_utilities(
    self, alternative: ExpansionAlternative
) -> Dict[str, float]:
    """Return the utility calculated for each stakeholder group."""
    return {
        stakeholder.name: self.calculate_stakeholder_utility(
            alternative, stakeholder
        )
        for stakeholder in self.stakeholders
    }


def optimize_value_aligned(self) -> Tuple[ExpansionAlternative, Dict]:
    """Rank the generated alternatives using utility and ethical scores."""
    if not self.alternatives:
        raise ValueError("Generate alternatives before running the optimization.")

    optimization_results = []
    for alternative in self.alternatives:
        stakeholder_utilities = self._analyze_stakeholder_utilities(alternative)
        total_utility = sum(stakeholder_utilities.values())
        ethical_compliance = (
            alternative.environmental_justice_score
            + alternative.labor_equity_score
            + alternative.intergenerational_score
            + alternative.transparency_score
        ) / 4
        combined_score = total_utility * (0.7 + 0.3 * ethical_compliance)

        optimization_results.append(
            {
                "alternative": alternative.name,
                "total_utility": total_utility,
                "ethical_compliance": ethical_compliance,
                "combined_score": combined_score,
                "npv": alternative.npv,
                "investment": alternative.investment_cost,
                "job_displacement": alternative.job_displacement,
                "community_benefits": alternative.community_benefits,
            }
        )

    best_score = max(result["combined_score"] for result in optimization_results)
    tied_alternatives = [
        result["alternative"]
        for result in optimization_results
        if math.isclose(
            result["combined_score"], best_score, rel_tol=1e-9, abs_tol=1e-9
        )
    ]
    best_alternative = next(
        alternative
        for alternative in self.alternatives
        if alternative.name == tied_alternatives[0]
    )

    return best_alternative, {
        "optimization_results": optimization_results,
        "best_score": best_score,
        "tied_alternatives": tied_alternatives,
        "stakeholder_utilities": self._analyze_stakeholder_utilities(
            best_alternative
        ),
    }


ValueAlignedOptimizer.calculate_stakeholder_utility = (
    calculate_stakeholder_utility
)
ValueAlignedOptimizer._analyze_stakeholder_utilities = (
    _analyze_stakeholder_utilities
)
ValueAlignedOptimizer.optimize_value_aligned = optimize_value_aligned

print("Stakeholder utility and alternative-ranking methods are ready.")

## 6. Inspect the inputs and ethical scores

This cell creates the optimizer, displays the stakeholder preferences and policy parameters, generates the alternatives, and reports the inputs used for each option. The overview is an input check: it helps readers see what the model is comparing before interpreting its ranking.

In [ ]:
print("\n" + "=" * 80)
print("VALUE-ALIGNED AND ETHICAL FRAMEWORK: PORT CAPACITY PLANNING")
print("=" * 80)

optimizer = ValueAlignedOptimizer()

print("\nSTAKEHOLDER GROUPS")
for stakeholder in optimizer.stakeholders:
    print(f"\n{stakeholder.name}:")
    print(f"  Economic weight: {stakeholder.economic_weight:.0%}")
    print(f"  Environmental weight: {stakeholder.environmental_weight:.0%}")
    print(f"  Social weight: {stakeholder.social_weight:.0%}")
    print(f"  Risk-tolerance parameter: {stakeholder.risk_tolerance:.0%}")
    print(f"  Time-preference rate: {stakeholder.time_preference:.0%}")

print("\nPOLICY PARAMETERS")
print(
    "  Environmental-justice impact ratio: "
    f"{optimizer.constraints.environmental_justice_ratio:.0%}"
)
print(
    "  Labor displacement ratio: "
    f"{optimizer.constraints.labor_displacement_ratio:.0%}"
    " of jobs created, plus the training proxy"
)
print(
    "  Intergenerational current-cost ratio: "
    f"{optimizer.constraints.intergenerational_debt_ratio:.0%}"
    " of future-benefit proxy"
)
print(
    "  Community-benefit ratio, wage standard, and carbon-neutrality year "
    "are not scored because required supporting data are absent."
)

alternatives = optimizer.generate_alternatives()
print(f"\nGenerated {len(alternatives)} illustrative alternatives.")

print("\nALTERNATIVE INPUTS AND ETHICAL SCORES")
for alternative in alternatives:
    ethical_score = (
        alternative.environmental_justice_score
        + alternative.labor_equity_score
        + alternative.intergenerational_score
        + alternative.transparency_score
    ) / 4
    print(f"\n{alternative.name}:")
    print(f"  Investment: ${alternative.investment_cost:,.0f} million")
    print(f"  Capacity increase: {alternative.capacity_increase:.0%}")
    print(f"  Technology: {alternative.technology_level}")
    print(f"  Implementation time: {alternative.implementation_years} years")
    print(f"  NPV: ${alternative.npv:,.0f} million")
    print(
        f"  Jobs created: {alternative.job_creation:,}; "
        f"jobs displaced: {alternative.job_displacement:,}"
    )
    print(
        f"  Carbon footprint: {alternative.carbon_footprint:,.0f} "
        "tonnes CO2 per year"
    )
    print(
        f"  Community benefits: ${alternative.community_benefits:,.0f} "
        "million per year"
    )
    print(f"  Mean ethical assessment score: {ethical_score:.1%}")

## 7. Run the value-aligned ranking and compare with the NPV choice

The NPV-only comparator selects the alternative with the largest NPV among the same options. Comparing it with the value-aligned selection shows how the chosen weighting and ethical score multiplier affect the decision. The reported differences describe this model's inputs and assumptions; they are not causal estimates of real project outcomes.

In [ ]:
best_alternative, analysis = optimizer.optimize_value_aligned()
utilities = analysis["stakeholder_utilities"]
economic_best = max(alternatives, key=lambda alternative: alternative.npv)

best_result = next(
    result
    for result in analysis["optimization_results"]
    if result["alternative"] == best_alternative.name
)
ethical_avg = best_result["ethical_compliance"]

print("\n" + "=" * 60)
print("VALUE-ALIGNED SELECTION")
print("=" * 60)
print(f"Selected alternative: {best_alternative.name}")
print(f"Combined score: {analysis['best_score']:.6f}")
print(f"Mean ethical assessment score: {ethical_avg:.1%}")
if len(analysis["tied_alternatives"]) > 1:
    print("Alternatives tied within numerical tolerance:")
    for name in analysis["tied_alternatives"]:
        print(f"  - {name}")

print("\nVALUE-ALIGNED ALTERNATIVE DETAILS")
print(f"  Investment: ${best_alternative.investment_cost:,.0f} million")
print(f"  Capacity increase: {best_alternative.capacity_increase:.0%}")
print(f"  Technology: {best_alternative.technology_level}")
print(f"  Implementation time: {best_alternative.implementation_years} years")
print(f"  NPV: ${best_alternative.npv:,.0f} million")
print(f"  Jobs created: {best_alternative.job_creation:,}")
print(f"  Jobs displaced: {best_alternative.job_displacement:,}")
print(
    f"  Carbon footprint: {best_alternative.carbon_footprint:,.0f} "
    "tonnes CO2 per year"
)

print("\nETHICAL ASSESSMENT SCORES")
print(
    "  Environmental justice: "
    f"{best_alternative.environmental_justice_score:.1%}"
)
print(f"  Labor equity: {best_alternative.labor_equity_score:.1%}")
print(
    "  Intergenerational equity: "
    f"{best_alternative.intergenerational_score:.1%}"
)
print(f"  Transparency proxy: {best_alternative.transparency_score:.1%}")

print("\nNPV-ONLY COMPARATOR")
print(f"  Alternative: {economic_best.name}")
print(f"  NPV: ${economic_best.npv:,.0f} million")
print(f"  Jobs displaced: {economic_best.job_displacement:,}")
print(
    f"  Carbon footprint: {economic_best.carbon_footprint:,.0f} "
    "tonnes CO2 per year"
)

npv_tradeoff = (
    (best_alternative.npv - economic_best.npv) / economic_best.npv * 100
    if not math.isclose(economic_best.npv, 0.0)
    else float("nan")
)
job_displacement_reduction = (
    (economic_best.job_displacement - best_alternative.job_displacement)
    / economic_best.job_displacement
    * 100
    if economic_best.job_displacement != 0
    else float("nan")
)
carbon_reduction = (
    (economic_best.carbon_footprint - best_alternative.carbon_footprint)
    / economic_best.carbon_footprint
    * 100
    if economic_best.carbon_footprint != 0
    else float("nan")
)

print("\nCOMPARISON WITH THE NPV-ONLY ALTERNATIVE")
print(f"  NPV difference: {npv_tradeoff:.1f}% relative to the NPV-only choice")
print(
    "  Job-displacement difference: "
    f"{job_displacement_reduction:.1f}% reduction"
)
print(f"  Carbon-footprint difference: {carbon_reduction:.1f}% reduction")

print("\nSTAKEHOLDER UTILITY FOR THE VALUE-ALIGNED SELECTION")
for stakeholder_name, utility in utilities.items():
    print(f"  {stakeholder_name}: {utility:.3f} USD-million-equivalent units")

## 8. Visualize the trade-offs

The plots display different views of the same alternatives: NPV against the mean ethical score, stakeholder utilities for the selected option, its four ethical assessment scores, and investment against annual community benefits. The star marks the value-aligned selection. These are descriptive plots; they do not add constraints or change the ranking.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Value-Aligned Port Expansion: Alternative Comparisons",
    fontsize=16,
    fontweight="bold",
)

# Compare economic return with the mean ethical assessment score.
ethical_scores = [
    (
        alternative.environmental_justice_score
        + alternative.labor_equity_score
        + alternative.intergenerational_score
        + alternative.transparency_score
    )
    / 4
    for alternative in alternatives
]
npvs = [alternative.npv for alternative in alternatives]
axes[0, 0].scatter(npvs, ethical_scores, s=100, alpha=0.8, color="steelblue")
axes[0, 0].scatter(
    best_alternative.npv,
    ethical_avg,
    s=260,
    color="red",
    marker="*",
    edgecolors="black",
    linewidth=1,
    label="Value-aligned selection",
)
for alternative, score in zip(alternatives, ethical_scores):
    axes[0, 0].annotate(
        alternative.name,
        (alternative.npv, score),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
axes[0, 0].set_xlabel("NPV (USD millions)")
axes[0, 0].set_ylabel("Mean ethical assessment score (0-1)")
axes[0, 0].set_title("NPV and Ethical Assessment")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Compare the utilities calculated for each stakeholder group.
stakeholder_names = list(utilities.keys())
stakeholder_values = list(utilities.values())
bars = axes[0, 1].bar(
    stakeholder_names,
    stakeholder_values,
    color="steelblue",
)
axes[0, 1].set_title("Stakeholder Utility for the Selected Alternative")
axes[0, 1].set_ylabel("Utility (USD-million-equivalent units)")
axes[0, 1].tick_params(axis="x", rotation=35)
axes[0, 1].grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, stakeholder_values):
    offset = 0.01 * max(abs(value), 1.0)
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

# Display the four components contributing to the mean ethical score.
compliance_categories = [
    "Environmental justice",
    "Labor equity",
    "Intergenerational equity",
    "Transparency proxy",
]
selected_scores = [
    best_alternative.environmental_justice_score,
    best_alternative.labor_equity_score,
    best_alternative.intergenerational_score,
    best_alternative.transparency_score,
]
bars = axes[1, 0].bar(
    compliance_categories,
    selected_scores,
    color=["seagreen", "cornflowerblue", "goldenrod", "salmon"],
)
axes[1, 0].set_title("Ethical Assessment of the Selected Alternative")
axes[1, 0].set_ylabel("Score (0-1)")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].tick_params(axis="x", rotation=25)
axes[1, 0].grid(True, axis="y", alpha=0.3)
for bar, score in zip(bars, selected_scores):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{score:.1%}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

# Compare investment with annual community benefits for all alternatives.
investments = [alternative.investment_cost for alternative in alternatives]
community_benefits = [
    alternative.community_benefits for alternative in alternatives
]
axes[1, 1].scatter(
    investments,
    community_benefits,
    s=100,
    alpha=0.8,
    color="steelblue",
)
axes[1, 1].scatter(
    best_alternative.investment_cost,
    best_alternative.community_benefits,
    s=260,
    color="red",
    marker="*",
    edgecolors="black",
    linewidth=1,
    label="Value-aligned selection",
)
for alternative in alternatives:
    axes[1, 1].annotate(
        alternative.name,
        (alternative.investment_cost, alternative.community_benefits),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
axes[1, 1].set_xlabel("Investment (USD millions)")
axes[1, 1].set_ylabel("Community benefits (USD millions per year)")
axes[1, 1].set_title("Investment and Annual Community Benefits")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

fig.tight_layout()
plt.show()
print("The comparison plots are complete.")

## 9. Interpret the results and identify decision limits

The final summary reports model-derived comparisons and the selected alternative's scores. A lower score on an ethical dimension signals a weaker result under that dimension's stated proxy; it does not establish legal compliance or ethical acceptability. The recommendations are therefore governance actions to consider alongside further evidence and stakeholder review, not conclusions proven by the numerical ranking.

In [ ]:
total_carbon = sum(alternative.carbon_footprint for alternative in alternatives)
average_carbon = total_carbon / len(alternatives)
carbon_difference = (
    economic_best.carbon_footprint - best_alternative.carbon_footprint
)
carbon_cost_difference_usd = carbon_difference * 50

net_jobs_selected = (
    best_alternative.job_creation - best_alternative.job_displacement
)
net_jobs_npv_only = economic_best.job_creation - economic_best.job_displacement
npv_difference = best_alternative.npv - economic_best.npv
investment_difference = (
    best_alternative.investment_cost - economic_best.investment_cost
)

print("\n" + "=" * 80)
print("VALUE-ALIGNED FRAMEWORK: MODEL SUMMARY")
print("=" * 80)

print("\nEnvironmental indicators:")
print(
    f"  Mean carbon footprint across the listed alternatives: "
    f"{average_carbon:,.0f} tonnes CO2 per year"
)
print(
    "  Carbon-footprint difference from the NPV-only choice: "
    f"{carbon_difference:,.0f} tonnes CO2 per year"
)
print(
    "  Illustrative annual carbon-cost difference at USD 50 per tonne: "
    f"${carbon_cost_difference_usd:,.0f}"
)

print("\nSocial indicators:")
print(f"  Net jobs for the value-aligned selection: {net_jobs_selected:,}")
print(f"  Net jobs for the NPV-only choice: {net_jobs_npv_only:,}")
print(
    f"  Training opportunities in the selected alternative: "
    f"{best_alternative.training_opportunities:,}"
)

print("\nEconomic comparison:")
print(f"  NPV difference: ${npv_difference:,.0f} million")
print(f"  Investment difference: ${investment_difference:,.0f} million")

print("\nEthical assessment of the selected alternative:")
print(f"  Mean score: {ethical_avg:.1%}")
print(
    "  Environmental justice proxy: "
    f"{best_alternative.environmental_justice_score:.1%}"
)
print(f"  Labor equity proxy: {best_alternative.labor_equity_score:.1%}")
print(
    "  Intergenerational equity proxy: "
    f"{best_alternative.intergenerational_score:.1%}"
)
print(f"  Transparency proxy: {best_alternative.transparency_score:.1%}")

print("\nDecision and governance notes:")
print(f"  Highest combined-score option: {best_alternative.name}")
print(
    "  Validate proxy rates, stakeholder weights, and impact estimates "
    "with affected groups and relevant evidence."
)
print(
    "  Assess wages, local-benefit shares, emissions over time, and "
    "distribution of impacts before treating policy standards as satisfied."
)
print(
    "  Monitor workforce transitions, environmental effects, and "
    "community outcomes throughout implementation."
)
print("=" * 80)